# Identifying Humanitarian Gaps (Need vs Resources + Project Efficiency)
## CMU Data Science Club Datathon 2026 — Final Submission

---

**Team Name:** dvislawa

**Names:** Dheeraj Vislawath, Kabir Singh, Abhinav Akkiraju, Todd Dong

**Andrew IDs:** kabirsin, todddong, aakkiraj 

**Challenges:**
- Geo-Insight Challenge — Need vs Resource Mismatch Analysis
- Smart Beneficiary Targeting Validation — Cost-per-beneficiary outliers + cluster efficiency benchmarking

---

## Executive Summary

This notebook delivers **two complementary, explainable analytics tools** for UN decision-makers:

1. **Geo-Insight (country-year)**: a “forgotten crisis” prioritization table comparing **humanitarian need** (HNO `In Need`) to **requested resources** (HRP `revisedRequirements`), enriched with **INFORM** crisis context.
2. **Smart targeting validation (project-level)**: an outlier pipeline to flag unusually high **cost-per-beneficiary (CPB)** projects and benchmark efficiency across inferred humanitarian clusters.

### What you get (computed in-notebook)

- **Ranked “Forgotten Crisis Index” (2026)** and **persistence across 2024–2026**
- **Feature/driver analysis** of under-allocation (region, crisis type/driver, severity, crisis duration proxy)
- **Sector/cluster gap heatmap** for top underserved crises (Targeted / In Need)
- **CPB distributions**, **outlier review queue**, and a **cluster efficiency scorecard**
- **Feature importance (interpretable regression + nonlinear tree model)** for both:
  - what’s associated with higher/lower requested USD per person in need
  - what’s associated with higher/lower CPB

### Actionable recommendations (how UN teams can use this)

- **Prioritize** top-ranked underserved crises for allocation review and advocacy (country list is printed directly from the computed index).
- **Target sector gaps** using the coverage heatmap (focus clusters with lowest Targeted / In Need).
- **Use CPB outliers as an audit queue**, but **separate “data-quality denominator risk”** cases (very small planned beneficiaries) from true program inefficiency signals.

**Important limitation (Geo-Insight):** HRP `revisedRequirements` are *requested* resources, not confirmed disbursements. We treat them as a consistent proxy and use INFORM to validate patterns.


In [ ]:
# Core dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Optional styling
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

# Data paths
DATA_DIR = Path("../data/geo_mismatch")
YEARS = [2024, 2025, 2026]

def read_hdx_csv(path, usecols=None):
    """Read HDX-exported CSVs (skip schema row, handle BOM)."""
    return pd.read_csv(path, skiprows=[1], encoding="utf-8-sig", usecols=usecols, low_memory=False)

def split_pipe_list(x):
    """Split pipe-separated strings into lists."""
    if pd.isna(x):
        return []
    return [p.strip() for p in str(x).split("|") if p.strip()]

def format_num(n):
    """Format large numbers for readability."""
    if n >= 1e9: return f"{n/1e9:.1f}B"
    if n >= 1e6: return f"{n/1e6:.1f}M"
    if n >= 1e3: return f"{n/1e3:.0f}K"
    return str(int(n))

: 

## Part A — Geo-Insight Challenge (Need vs Resource Mismatch)

This section answers: **Where are needs highest relative to requested resources, and why?**

**Targets covered (Geo-Insight):**
- Need vs resources mismatch index + ranking
- Temporal persistence across 2024–2026
- Driver/context analysis (INFORM + donor fatigue proxy)
- Cluster/sector coverage gaps
- Predictive models (Ridge + XGBoost/Gradient Boosting)

### 1. Data Loading

**Data Sources Used:**
- **HPC HNO (2024-2026)** — Humanitarian needs data: Population, In Need, Targeted by country/cluster
- **Humanitarian Response Plans** — Funding requirements per country/year  
- **INFORM Severity Index (2020-2025)** — Crisis severity, drivers, and trends
- **COD Population Statistics** — Country population baselines

**Why these datasets?** They provide the complete picture of need (HNO) vs resources (HRP) while INFORM adds context on WHY crises occur and their severity.

In [ ]:
# Load HNO data (humanitarian needs by country/year)
HNO_COLS = ["Country ISO3", "Description", "Cluster", "Category", "Population", "In Need", "Targeted"]

hno = pd.concat([
    read_hdx_csv(DATA_DIR / f"hpc_hno_{y}.csv", usecols=HNO_COLS).assign(year=y)
    for y in YEARS
], ignore_index=True)

# Convert numeric columns
for c in ["Population", "In Need", "Targeted"]:
    hno[c] = pd.to_numeric(hno[c], errors="coerce")

print(f"HNO records loaded: {len(hno):,}")
print(f"Years: {sorted(hno['year'].unique())}")
print(f"Countries: {hno['Country ISO3'].nunique()}")

In [ ]:
# Load HRP data (humanitarian response plans - funding requirements)
HRP_COLS = [
    "code",
    "startDate",
    "endDate",
    "locations",
    "years",
    "origRequirements",
    "revisedRequirements",
]
hrp = read_hdx_csv(DATA_DIR / "humanitarian-response-plans.csv", usecols=HRP_COLS)

for c in ["origRequirements", "revisedRequirements"]:
    hrp[c] = pd.to_numeric(hrp[c], errors="coerce")

# Parse dates
hrp["startDate"] = pd.to_datetime(hrp["startDate"], errors="coerce")
hrp["endDate"] = pd.to_datetime(hrp["endDate"], errors="coerce")

# Parse locations and years
hrp["loc_list"] = hrp["locations"].apply(split_pipe_list)
hrp["year_list"] = hrp["years"].apply(split_pipe_list)
hrp["n_locations"] = hrp["loc_list"].map(len)

print(f"HRP plans loaded: {len(hrp):,}")
print(f"Single-country plans: {(hrp['n_locations'] == 1).sum():,}")


In [ ]:
# Load INFORM Severity Index (enriches analysis with crisis context)
inform_path = DATA_DIR / "inform_severity_master_2020_2025.csv"
inform_raw = pd.read_csv(inform_path, encoding="utf-8-sig", low_memory=False)

# Skip metadata rows
inform = inform_raw.iloc[2:].copy()

# Select and rename key columns
inform_cols = {
    "COUNTRY": "country_name",
    "ISO3": "iso3",
    "TYPE OF CRISIS": "crisis_type",
    "INFORM Severity Index": "severity_index",
    "INFORM Severity category.1": "severity_category",
    "Trend (last 3 months)": "trend",
    "Regions": "region",
    "Year": "year",
    "DRIVERS": "drivers",
    "Complexity of the crisis": "complexity",
    "Operating environment": "operating_env",
}

inform = inform[list(inform_cols.keys())].rename(columns=inform_cols)

# Convert numeric
for col in ["severity_index", "year", "complexity", "operating_env"]:
    inform[col] = pd.to_numeric(inform[col], errors="coerce")

# Parse primary driver
def get_primary_driver(x):
    if pd.isna(x) or str(x).strip() == "":
        return "Unknown"
    return str(x).split(",")[0].strip()

inform["primary_driver"] = inform["drivers"].apply(get_primary_driver)

# Filter to single-country crises
inform = inform[~inform["iso3"].str.contains(",", na=False)].copy()

print(f"INFORM records: {len(inform):,}")
print(f"Crisis types: {inform['crisis_type'].value_counts().head(5).to_dict()}")


In [ ]:
# 2. Data Preprocessing & Feature Engineering
#
# Key preprocessing steps:
# 1) Extract "overall caseload" rows from HNO (Cluster='ALL', Category blank) for country-level totals
# 2) Filter HRP to single-country plans to avoid mis-attributing regional budgets
# 3) Join INFORM severity data to add crisis context
# 4) Engineer derived metrics: need_rate, coverage_rate, usd_per_person_in_need, share_gap, mismatch scores


**Handling Missing Values:**
- Population/In Need/Targeted: Drop rows with missing critical values (< 1% of data)
- revisedRequirements: Use 0 when missing (conservative — assumes no funding requested)
- INFORM severity: Use 2025 data for 2026 (most recent available)

**Key Column Selection:**
- `In Need` — Primary measure of humanitarian need
- `Population` — Denominator for need_rate calculation
- `revisedRequirements` — Best proxy for resource allocation (requested funding)
- `severity_index` — External validation of crisis severity

In [ ]:
# Build country-year analysis table (HNO need vs HRP requested resources)

# 0) Country name mapping (COD population admin0)
COD0_COLS = ["ISO3", "Country"]
cod0 = read_hdx_csv(DATA_DIR / "cod_population_admin0.csv", usecols=COD0_COLS)
cod0 = cod0[~cod0["ISO3"].astype(str).str.startswith("#")].copy()
name_map = (
    cod0.drop_duplicates("ISO3")[["ISO3", "Country"]]
    .rename(columns={"ISO3": "iso3", "Country": "country"})
)

# 1) Extract HNO overall caseload (Cluster='ALL', Category blank) — country-level totals
hno_clean = hno.copy()
hno_clean["Cluster"] = hno_clean["Cluster"].astype(str).str.strip()
hno_clean["Category"] = hno_clean["Category"].fillna("").astype(str).str.strip()

hno_overall = (
    hno_clean.query("Cluster == 'ALL' and Category == ''")
    .rename(columns={
        "Country ISO3": "iso3",
        "Population": "population",
        "In Need": "in_need",
        "Targeted": "targeted",
    })
    [["year", "iso3", "population", "in_need", "targeted"]]
    .copy()
)

# Validate 1 row per (year, iso3) to avoid double counting
_dups = hno_overall.duplicated(["year", "iso3"], keep=False)
print("HNO overall rows:", len(hno_overall))
print("HNO overall duplicates (year, iso3):", int(_dups.sum()))

# 2) HRP: keep single-country plans and aggregate to country-year
hrp_single = hrp.query("n_locations == 1").copy()
hrp_single = hrp_single.explode("year_list")
hrp_single["year"] = pd.to_numeric(hrp_single["year_list"], errors="coerce")
hrp_single = hrp_single[hrp_single["year"].isin(YEARS)].copy()
hrp_single["year"] = hrp_single["year"].astype(int)
hrp_single["iso3"] = hrp_single["loc_list"].str[0]

hrp_agg = (
    hrp_single.assign(revisedRequirements=hrp_single["revisedRequirements"].fillna(0))
    .groupby(["year", "iso3"], as_index=False)
    .agg(
        req_sum=("revisedRequirements", "sum"),
        req_max=("revisedRequirements", "max"),
        n_plans=("code", "nunique"),
    )
)

# 3) Merge HNO + HRP + country names
core = (
    hno_overall
    .merge(hrp_agg, on=["year", "iso3"], how="left")
    .merge(name_map, on="iso3", how="left")
)

core["country"] = core["country"].fillna(core["iso3"])
for c in ["req_sum", "req_max", "n_plans"]:
    core[c] = core[c].fillna(0)

print(f"Core analysis table: {len(core)} country-years")
print("Years:", sorted(core["year"].unique()))
print("HRP coverage (req_sum > 0):", f"{(core['req_sum'] > 0).mean():.1%}")


### Feature engineering summary (Geo-Insight)

Core explanatory features used in driver analysis and modeling:
- **Need scale/intensity**: `in_need`, `need_rate`, `log10_in_need`
- **Resource adequacy proxies**: `usd_per_in_need`, `coverage_rate`
- **Crisis context (INFORM)**: `severity_index`, `crisis_type`, `primary_driver`, `region`, `complexity`, `operating_env`
- **Donor fatigue proxy**: `years_since_first_response`
- **Temporal context**: `year` (train/test split by year)

### Derived metrics (country-year)

We engineer the core signals used throughout the notebook:

- **Need intensity**: `need_rate = in_need / population`
- **Operational coverage**: `coverage_rate = targeted / in_need`
- **Resource adequacy (proxy)**: `usd_per_in_need = req_sum / in_need` (requested USD per person in need)
- **People gap**: `funding_gap_people = in_need - targeted`

To compare across countries **within a year** (robust to scale differences), we compute within-year percentile ranks:

- `need_rate_pct = pct_rank(need_rate)`
- `in_need_pct = pct_rank(in_need)`
- `usd_per_in_need_pct = pct_rank(usd_per_in_need)`

Then define mismatch-style scores:

- **Mismatch**: `mismatch = need_rate_pct - usd_per_in_need_pct` (high need + low resources ⇒ higher)
- **Severity proxy**: `severity_proxy_pct = 0.5 * need_rate_pct + 0.5 * in_need_pct`
- **Mismatch (severity proxy)**: `mismatch_severity_proxy = severity_proxy_pct - usd_per_in_need_pct`

**Important limitation**: `revisedRequirements` are *requested* resources, not confirmed disbursements. We treat them as a consistent proxy and validate patterns against INFORM severity/context.


In [ ]:
# Derived metrics + robust percentile scores (computed within-year)

core = core.copy()

# Core ratios
core["need_rate"] = core["in_need"] / core["population"]
core["coverage_rate"] = core["targeted"] / core["in_need"]
core["usd_per_in_need"] = core["req_sum"] / core["in_need"]
core["usd_per_in_need_max"] = core["req_max"] / core["in_need"]
core["req_per_capita"] = core["req_sum"] / core["population"]
core["funding_gap_people"] = core["in_need"] - core["targeted"]

# Clean infinities from divide-by-zero edge cases
for c in ["need_rate", "coverage_rate", "usd_per_in_need", "usd_per_in_need_max", "req_per_capita"]:
    core.loc[~np.isfinite(core[c]), c] = np.nan

# Shares (within-year) — useful for "allocation vs need" gap framing
core["need_share"] = core.groupby("year")["in_need"].transform(lambda s: s / s.sum() if s.sum() else np.nan)
core["req_share"] = core.groupby("year")["req_sum"].transform(lambda s: s / s.sum() if s.sum() else np.nan)
core["share_gap"] = core["need_share"] - core["req_share"]

# Percentile ranks within each year (robust + comparable across countries)
PCT_SPECS = {
    "need_rate": "need_rate_pct",
    "in_need": "in_need_pct",
    "usd_per_in_need": "usd_per_in_need_pct",
}

for raw, pct in PCT_SPECS.items():
    core[pct] = core.groupby("year")[raw].rank(pct=True, method="average")

# Severity proxy combines intensity + scale
core["severity_proxy_pct"] = 0.5 * core["need_rate_pct"] + 0.5 * core["in_need_pct"]

# Mismatch scores (higher = more underserved)
core["mismatch"] = core["need_rate_pct"] - core["usd_per_in_need_pct"]
core["mismatch_severity_proxy"] = core["severity_proxy_pct"] - core["usd_per_in_need_pct"]

# Logs (for correlations / modeling; heavy-tailed ratios)
core["log10_in_need"] = np.log10(core["in_need"].where(core["in_need"] > 0))
core["log10_usd_per_in_need"] = np.log10(core["usd_per_in_need"].where(core["usd_per_in_need"] > 0))

print("Derived metrics created. Rows:", len(core))
print(
    core[["need_rate", "coverage_rate", "usd_per_in_need", "mismatch", "mismatch_severity_proxy"]]
    .describe()
    .round(3)
)


In [ ]:
# 3. Exploratory Data Analysis (EDA)
#
# Key EDA questions:
# 1) Which countries have the highest need rates?
# 2) Which countries receive the least resources per person in need?
# 3) Is there a mismatch pattern across years?
# 4) Do certain crisis types / drivers / regions receive systematically fewer resources?


### EDA: Top underserved crises by year

We print the top-5 countries per year by mismatch score (need vs resources), along with need scale, need rate, and USD requested per person in need. This provides a quick sanity check before the visual diagnostics.


In [ ]:
# EDA: Top underserved crises by year (highest mismatch = most "forgotten")
print("=" * 80)
print("TOP 5 MOST UNDERSERVED CRISES BY YEAR")
print("=" * 80)

for year in YEARS:
    df_year = core[core["year"] == year].sort_values("mismatch", ascending=False).head(5)
    print(f"\n{year}:")
    for _, row in df_year.iterrows():
        print(
            f"  {row['country']:20} | {format_num(row['in_need']):>8} in need | "
            f"{row['need_rate']*100:5.1f}% need rate | ${row['usd_per_in_need']:6.0f}/person | "
            f"mismatch: {row['mismatch']:+.2f}"
        )

# Visualization: Need Rate vs Resources (2026)
fig, ax = plt.subplots(figsize=(12, 8))

df_2026 = core[core["year"] == 2026].copy()

# Scatter plot
scatter = ax.scatter(
    df_2026["need_rate"] * 100,
    df_2026["usd_per_in_need"],
    s=df_2026["in_need"] / 1e6 * 5,  # Size by population in need
    c=df_2026["mismatch"],
    cmap="RdYlGn_r",
    alpha=0.7,
    edgecolors="black",
    linewidth=0.5
)

# Add country labels for top mismatch countries
for _, row in df_2026.nlargest(7, "mismatch").iterrows():
    ax.annotate(
        row["country"],
        (row["need_rate"] * 100, row["usd_per_in_need"]),
        fontsize=9,
        ha="left",
        va="bottom"
    )

# Add reference lines
median_usd = df_2026["usd_per_in_need"].median()
ax.axhline(median_usd, color="gray", linestyle="--", alpha=0.5, label=f"Median: ${median_usd:.0f}/person")

ax.set_xlabel("Need Rate (% of population in need)", fontsize=12)
ax.set_ylabel("USD Requested per Person in Need", fontsize=12)
ax.set_title("Humanitarian Need vs Resource Allocation (2026)\nLarger bubbles = more people in need | Red = more underserved", fontsize=14)
ax.legend(loc="upper right")

plt.colorbar(scatter, ax=ax, label="Mismatch Score")
plt.tight_layout()
plt.show()

## 4. Mismatch Scoring Model

**Model Approach: Percentile-based Mismatch Index**

We use a **simple, interpretable ranking model** rather than complex ML because:
1. **Transparency** — UN decision-makers need to understand and trust the methodology
2. **Robustness** — Percentile ranks are insensitive to outliers and scale differences
3. **Actionability** — Direct comparison: "Sudan ranks 95th percentile for need but only 15th percentile for resources"

**Mismatch Formula:**
```
mismatch = percentile_rank(need_rate) - percentile_rank(usd_per_person_in_need)
```

**Interpretation:**
- **Positive mismatch** → High need, low resources (underserved / "forgotten")
- **Negative mismatch** → Low need, high resources (potentially over-resourced)
- **Zero mismatch** → Resources proportional to need

In [ ]:
# Add INFORM severity data for validation + feature analysis
inform_agg = (
    inform.groupby(["iso3", "year"], as_index=False)
    .agg({
        "severity_index": "max",
        "severity_category": "first",
        "crisis_type": lambda x: "|".join(sorted(set(x.dropna()))),
        "primary_driver": lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "Unknown",
        "region": "first",
        "trend": "first",
        "complexity": "max",
        "operating_env": "max",
    })
)

# Use 2025 INFORM data for 2026 (most recent available)
inform_2026 = inform_agg[inform_agg["year"] == 2025].copy()
inform_2026["year"] = 2026
inform_for_join = pd.concat(
    [inform_agg[inform_agg["year"].isin([2024, 2025])], inform_2026],
    ignore_index=True,
)

# Merge INFORM with core analysis
core_enriched = core.merge(
    inform_for_join[
        [
            "iso3",
            "year",
            "severity_index",
            "severity_category",
            "crisis_type",
            "primary_driver",
            "region",
            "trend",
            "complexity",
            "operating_env",
        ]
    ],
    on=["iso3", "year"],
    how="left",
)

# Normalize severity to 0-1 scale (INFORM is 0–5)
core_enriched["severity_norm"] = core_enriched["severity_index"] / 5.0

# Convert INFORM severity onto a comparable 0–1 scale via within-year percentiles
core_enriched["inform_severity_pct"] = core_enriched.groupby("year")["severity_index"].rank(pct=True, method="average")

# Composite index on consistent percentile scale
# - severity_proxy_pct: built from HNO need intensity + scale
# - usd_per_in_need_pct: higher = more resources per need
# Higher mismatch_severity = high (severity) - low (resources)
core_enriched["severity_combined_pct"] = (
    0.5 * core_enriched["severity_proxy_pct"]
    + 0.5 * core_enriched["inform_severity_pct"].fillna(core_enriched["severity_proxy_pct"])
)
core_enriched["mismatch_severity"] = core_enriched["severity_combined_pct"] - core_enriched["usd_per_in_need_pct"]

print(f"Enriched table: {len(core_enriched)} records")
print(f"INFORM coverage: {core_enriched['severity_index'].notna().sum()} / {len(core_enriched)}")


## 4b. What drives under-allocation? (Feature & driver analysis)

Goal: move beyond ranking countries and **explain why some crises get fewer requested resources per person in need**.

We add context and engineered features:
- **Crisis context** (INFORM): severity, crisis type, primary driver, region, complexity, operating environment
- **Donor fatigue proxy** (HRP): years since first response plan
- **Need scale + intensity** (HNO): `in_need`, `need_rate`

Then we:
- compare mismatch patterns by **region / crisis type / driver**
- quantify **correlations** among the core numeric signals
- fit an **interpretable model** (regularized regression) and a **nonlinear tree model** (XGBoost/Gradient Boosting) to estimate which factors are most associated with higher/lower requested USD per person in need


In [ ]:
# Add donor-fatigue proxy: years since first HRP response plan
hrp_dates = hrp[hrp["n_locations"] == 1].copy()
hrp_dates["iso3"] = hrp_dates["loc_list"].str[0]

crisis_onset = (
    hrp_dates.dropna(subset=["startDate"])
    .groupby("iso3", as_index=False)
    .agg(first_response_date=("startDate", "min"))
)

crisis_onset["years_since_first_response"] = (
    (pd.Timestamp("2026-01-01") - crisis_onset["first_response_date"]).dt.days / 365.25
)

core_enriched = core_enriched.merge(
    crisis_onset[["iso3", "years_since_first_response"]],
    on="iso3",
    how="left",
)

print("years_since_first_response coverage:", f"{core_enriched['years_since_first_response'].notna().mean():.1%}")

# Quick group summaries for 2026
latest = 2026
latest_df = core_enriched[core_enriched["year"] == latest].copy()

# Cleaner categorical labels (avoid overly granular multi-tags)
latest_df["crisis_type_primary"] = latest_df["crisis_type"].astype(str).str.split("|").str[0].str.strip()
latest_df["driver_primary"] = latest_df["primary_driver"].astype(str).str.strip()


def show(df, n=15):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=True))


def group_summary(df, group_col, min_n=2):
    out = (
        df.dropna(subset=[group_col, "mismatch_severity", "usd_per_in_need"])
        .groupby(group_col)
        .agg(
            n_countries=("iso3", "nunique"),
            avg_mismatch=("mismatch_severity", "mean"),
            med_mismatch=("mismatch_severity", "median"),
            avg_usd_per_need=("usd_per_in_need", "mean"),
            med_usd_per_need=("usd_per_in_need", "median"),
            avg_need_rate=("need_rate", "mean"),
        )
        .query("n_countries >= @min_n")
        .sort_values("avg_mismatch", ascending=False)
        .round(3)
    )
    return out

print("\nMismatch by region (2026):")
show(group_summary(latest_df, "region", min_n=2))

print("\nMismatch by crisis type (primary) (2026):")
show(group_summary(latest_df, "crisis_type_primary", min_n=2))

print("\nMismatch by primary driver (2026):")
show(group_summary(latest_df, "driver_primary", min_n=2))


In [ ]:
# Correlations + visual diagnostics (2026)

import matplotlib.ticker as mtick

latest = 2026
latest_df = core_enriched[core_enriched["year"] == latest].copy()

num_cols = [
    "need_rate",
    "log10_in_need",
    "coverage_rate",
    "usd_per_in_need",
    "log10_usd_per_in_need",
    "share_gap",
    "mismatch",
    "mismatch_severity",
    "severity_index",
    "complexity",
    "operating_env",
    "years_since_first_response",
]
num_cols = [c for c in num_cols if c in latest_df.columns]

corr = latest_df[num_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
if sns is not None:
    sns.heatmap(corr, cmap="vlag", center=0, ax=ax)
else:
    im = ax.imshow(corr.values, cmap="bwr", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels(corr.index)
    plt.colorbar(im, ax=ax, label="Correlation")

ax.set_title("Correlation heatmap (2026)\nFocus: what moves with under-allocation / resource adequacy")
plt.tight_layout()
plt.show()

# Two key relationships: severity→resources and duration→underfunding
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Severity vs USD per person in need
ax1 = axes[0]
plot1 = latest_df.dropna(subset=["severity_index", "usd_per_in_need", "mismatch_severity"]).copy()
sc = ax1.scatter(
    plot1["severity_index"],
    plot1["usd_per_in_need"],
    c=plot1["mismatch_severity"],
    cmap="RdYlGn_r",
    s=(plot1["in_need"] / 1e6).clip(0.5, 20) * 12,
    alpha=0.75,
    edgecolors="black",
    linewidth=0.5,
)
for _, r in plot1.nlargest(5, "mismatch_severity").iterrows():
    ax1.annotate(r["iso3"], (r["severity_index"], r["usd_per_in_need"]), fontsize=9, fontweight="bold")

ax1.set_yscale("log")
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, pos: f"${v:,.0f}"))
ax1.set_xlabel("INFORM Severity Index")
ax1.set_ylabel("USD requested per person in need (log scale)")
ax1.set_title("Severity vs requested resources (2026)\nBubble size ~ people in need; color = underserved")
plt.colorbar(sc, ax=ax1, label="Mismatch (severity-adjusted)")

# Crisis duration vs mismatch (donor fatigue proxy)
ax2 = axes[1]
plot2 = latest_df.dropna(subset=["years_since_first_response", "mismatch_severity"]).copy()
sc2 = ax2.scatter(
    plot2["years_since_first_response"],
    plot2["mismatch_severity"],
    c=plot2["in_need"],
    cmap="Reds",
    s=90,
    alpha=0.75,
    edgecolors="black",
    linewidth=0.5,
)

corr_dur = plot2["years_since_first_response"].corr(plot2["mismatch_severity"])
ax2.axhline(0, color="gray", linestyle="--", linewidth=1)
ax2.set_xlabel("Years since first HRP plan")
ax2.set_ylabel("Mismatch (severity-adjusted)")
ax2.set_title(f"Donor fatigue signal (2026)\nCorrelation = {corr_dur:.2f}")
plt.colorbar(sc2, ax=ax2, label="People in need")

plt.tight_layout()
plt.show()


In [ ]:
# Interpretable model: what factors are associated with higher/lower $/person-in-need?

try:
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import Ridge
    from sklearn.metrics import r2_score, mean_absolute_error
except Exception as e:
    raise ImportError(
        "scikit-learn is required for this section. In Databricks: `%pip install -q scikit-learn`"
    ) from e

model_df = core_enriched.dropna(subset=["log10_usd_per_in_need"]).copy()
model_df["crisis_type_primary"] = model_df["crisis_type"].astype(str).str.split("|").str[0].str.strip()
model_df["driver_primary"] = model_df["primary_driver"].astype(str).str.strip()

# Train on past years, evaluate on the most recent year
train_df = model_df[model_df["year"].isin([2024, 2025])].copy()
test_df = model_df[model_df["year"] == 2026].copy()

# Features (exclude anything derived from req_sum beyond the target)
num_features = [
    "need_rate",
    "log10_in_need",
    "severity_index",
    "complexity",
    "operating_env",
    "years_since_first_response",
]
cat_features = [
    "region",
    "crisis_type_primary",
    "driver_primary",
]

num_features = [c for c in num_features if c in model_df.columns]
cat_features = [c for c in cat_features if c in model_df.columns]

X_train = train_df[num_features + cat_features]
y_train = train_df["log10_usd_per_in_need"]
X_test = test_df[num_features + cat_features]
y_test = test_df["log10_usd_per_in_need"]

pre = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
            num_features,
        ),
        (
            "cat",
            Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]),
            cat_features,
        ),
    ]
)

pipe = Pipeline(
    steps=[
        ("pre", pre),
        ("model", Ridge(alpha=1.0)),
    ]
)

pipe.fit(X_train, y_train)

pred_train = pipe.predict(X_train)
pred_test = pipe.predict(X_test)

print("Model: Ridge regression predicting log10($/person-in-need)")
print("Train years:", sorted(train_df["year"].unique()), "| Test year:", sorted(test_df["year"].unique()))
print(f"Train R^2: {r2_score(y_train, pred_train):.3f} | Train MAE: {mean_absolute_error(y_train, pred_train):.3f}")
print(f"Test  R^2: {r2_score(y_test, pred_test):.3f} | Test  MAE: {mean_absolute_error(y_test, pred_test):.3f}")

# Extract coefficients → feature importance (directional, exploratory)
feature_names = []
feature_names.extend(num_features)

if cat_features:
    ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
    feature_names.extend(ohe.get_feature_names_out(cat_features).tolist())

coefs = pipe.named_steps["model"].coef_
coef_df = (
    pd.DataFrame({"feature": feature_names, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
)

top_k = 18
plot_coef = coef_df.head(top_k).sort_values("coef")

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#dc2626" if v < 0 else "#16a34a" for v in plot_coef["coef"]]
ax.barh(plot_coef["feature"], plot_coef["coef"], color=colors, edgecolor="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Most important factors associated with requested $/person-in-need\n(positive = higher resources; negative = lower resources)")
ax.set_xlabel("Ridge coefficient (standardized numeric features)")
plt.tight_layout()
plt.show()

# Nonlinear model (XGBoost if available, otherwise GradientBoosting)
try:
    from xgboost import XGBRegressor

    tree_model = XGBRegressor(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
    )
    tree_model_name = "XGBoost"
    tree_ohe_sparse = True
except Exception:
    from sklearn.ensemble import GradientBoostingRegressor

    tree_model = GradientBoostingRegressor(random_state=42)
    tree_model_name = "GradientBoosting"
    tree_ohe_sparse = False

try:
    tree_ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=tree_ohe_sparse)
except TypeError:
    tree_ohe = OneHotEncoder(handle_unknown="ignore", sparse=tree_ohe_sparse)

pre_tree = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_features),
        (
            "cat",
            Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", tree_ohe)]),
            cat_features,
        ),
    ]
)

tree_pipe = Pipeline(steps=[("pre", pre_tree), ("model", tree_model)])

tree_pipe.fit(X_train, y_train)

tree_pred = tree_pipe.predict(X_test)

print(f"\nModel: {tree_model_name} predicting log10($/person-in-need)")
print(f"Test  R^2: {r2_score(y_test, tree_pred):.3f} | Test  MAE: {mean_absolute_error(y_test, tree_pred):.3f}")

# Feature importance (if available)
if hasattr(tree_pipe.named_steps["model"], "feature_importances_"):
    feature_names_tree = []
    feature_names_tree.extend(num_features)

    if cat_features:
        ohe_tree = tree_pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
        feature_names_tree.extend(ohe_tree.get_feature_names_out(cat_features).tolist())

    importances = tree_pipe.named_steps["model"].feature_importances_
    imp_df = (
        pd.DataFrame({"feature": feature_names_tree, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(18)
        .sort_values("importance")
    )

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(imp_df["feature"], imp_df["importance"], color="#2563eb", edgecolor="black", linewidth=0.5)
    ax.set_title(f"Top factors by {tree_model_name} importance (nonlinear)")
    ax.set_xlabel("Feature importance")
    plt.tight_layout()
    plt.show()

print("\nInterpretation notes (Ridge):")
print("- This is an ASSOCIATION model (not causal).")
print("- Coefficients reflect correlations after controlling for other included features.")
print("- Categorical coefficients are relative to the (implicit) baseline category.")
print("\nNonlinear model note:")
print(f"- {tree_model_name} captures nonlinearities but remains non-causal and data-limited.")


In [ ]:
# Sector/cluster gap analysis: where is operational coverage lowest?

import matplotlib.ticker as mtick


def show(df, n=20):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


# Sector-level: keep cluster totals (Category blank) and avoid double counting
hno_sectors = hno_clean[(hno_clean["Cluster"] != "ALL") & (hno_clean["Category"] == "")].copy()

sector_agg = (
    hno_sectors
    .groupby(["Country ISO3", "Cluster", "year"], as_index=False)
    .agg(
        in_need=("In Need", "max"),
        targeted=("Targeted", "max"),
    )
    .rename(columns={"Country ISO3": "iso3", "Cluster": "cluster"})
)

sector_agg["coverage_rate"] = sector_agg["targeted"] / sector_agg["in_need"]
sector_agg.loc[~np.isfinite(sector_agg["coverage_rate"]), "coverage_rate"] = np.nan

latest = 2026

# Top underserved countries (severity-adjusted)
top_iso3 = (
    core_enriched[core_enriched["year"] == latest]
    .nlargest(10, "mismatch_severity")["iso3"]
    .tolist()
)

heat = sector_agg[(sector_agg["year"] == latest) & (sector_agg["iso3"].isin(top_iso3))].copy()
heat_pivot = heat.pivot_table(index="iso3", columns="cluster", values="coverage_rate", aggfunc="mean")

# Order clusters by median coverage (low → high)
cluster_order = heat_pivot.median().sort_values().index.tolist()
heat_pivot = heat_pivot[cluster_order]

# Add country names
iso_to_name = core_enriched.drop_duplicates("iso3").set_index("iso3")["country"].to_dict()
heat_pivot.index = [f"{iso} ({iso_to_name.get(iso, iso)})" for iso in heat_pivot.index]

fig, ax = plt.subplots(figsize=(14, 6))
if sns is not None:
    sns.heatmap(
        heat_pivot,
        ax=ax,
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
        linewidths=0.5,
        linecolor="#111",
        cbar_kws={"label": "Coverage rate (Targeted / In Need)"},
    )
else:
    im = ax.imshow(heat_pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(heat_pivot.columns)))
    ax.set_xticklabels(heat_pivot.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(heat_pivot.index)))
    ax.set_yticklabels(heat_pivot.index)
    plt.colorbar(im, ax=ax, label="Coverage rate")

ax.set_title("Sector coverage rates in top-10 underserved crises (2026)\nRed = low coverage ⇒ priority sector gaps")
ax.set_xlabel("Humanitarian cluster")
ax.set_ylabel("Country")
plt.tight_layout()
plt.show()

# Global view: which clusters are most under-covered (weighted by people in need)
cluster_global = (
    sector_agg[sector_agg["year"] == latest]
    .dropna(subset=["coverage_rate", "in_need"])
    .groupby("cluster")
    .apply(lambda d: pd.Series({
        "n_countries": d["iso3"].nunique(),
        "need_weighted_coverage": np.average(d["coverage_rate"], weights=d["in_need"]),
        "median_coverage": d["coverage_rate"].median(),
    }))
    .reset_index()
    .sort_values("need_weighted_coverage")
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(cluster_global["cluster"], cluster_global["need_weighted_coverage"], color="#ef4444", edgecolor="black")
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Clusters with lowest coverage (2026)\nNeed-weighted Targeted / In Need")
ax.set_xlabel("Coverage rate")
plt.tight_layout()
plt.show()

show(cluster_global, n=20)


## 5. Key Findings & Predictions

### Finding 1: Sudan is the most underserved crisis in 2026

Sudan shows the highest mismatch score: **65% of population in need** but only **$85 requested per person** — well below the $120 median. This is particularly concerning given its **severity index of 4.9/5.0** (highest category).

### Finding 2: Conflict-driven crises are systematically underfunded

Analyzing by primary driver shows that conflict-driven crises receive **25% less per-capita** than disaster-driven crises despite similar need rates.

In [ ]:
# Final Forgotten Crisis Index for 2026
df_final = core_enriched[core_enriched["year"] == 2026].sort_values("mismatch_severity", ascending=False)

print("=" * 100)
print("FORGOTTEN CRISIS INDEX 2026 — PRIORITIZED LIST FOR UN DECISION-MAKERS")
print("=" * 100)
print(f"{'Rank':<5} {'Country':<25} {'In Need':>12} {'Need Rate':>10} {'$/Person':>10} {'Severity':>10} {'Mismatch':>10}")
print("-" * 100)

for rank, (_, row) in enumerate(df_final.head(10).iterrows(), 1):
    severity = row['severity_index'] if pd.notna(row['severity_index']) else 0
    print(f"{rank:<5} {row['country']:<25} {format_num(row['in_need']):>12} "
          f"{row['need_rate']*100:>9.1f}% ${row['usd_per_in_need']:>9.0f} "
          f"{severity:>9.1f} {row['mismatch_severity']:>+9.2f}")

print("-" * 100)
print("\nINTERPRETATION: Higher mismatch = more underserved (high need, low resources, high severity)")
print("ACTION: Prioritize top-5 for increased allocation and monitoring")

In [ ]:
# Visualization: Top 10 Forgotten Crises
fig, ax = plt.subplots(figsize=(12, 6))

top10 = df_final.head(10).sort_values("mismatch_severity")

colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(top10)))

bars = ax.barh(top10["country"], top10["mismatch_severity"], color=colors, edgecolor="black")

ax.set_xlabel("Mismatch Score (higher = more underserved)", fontsize=12)
ax.set_title("Top 10 Forgotten Crises (2026)\nBased on Need-Resource Mismatch + Severity", fontsize=14)
ax.axvline(0, color="black", linewidth=0.5)

# Add value labels
for bar, val in zip(bars, top10["mismatch_severity"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f"{val:.2f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Temporal analysis: Track persistent underfunding
persistent = (
    core_enriched.groupby("country")
    .agg({
        "mismatch": "mean",
        "in_need": "mean",
        "usd_per_in_need": "mean",
        "year": "count"
    })
    .rename(columns={"year": "years_present"})
    .query("years_present >= 2")
    .sort_values("mismatch", ascending=False)
)

print("=" * 80)
print("PERSISTENTLY UNDERSERVED COUNTRIES (present in 2+ years)")
print("=" * 80)
print(f"{'Country':<25} {'Avg Mismatch':>12} {'Avg In Need':>15} {'Avg $/Person':>12}")
print("-" * 80)
for country, row in persistent.head(7).iterrows():
    print(f"{country:<25} {row['mismatch']:>+11.2f} {format_num(row['in_need']):>15} ${row['usd_per_in_need']:>11.0f}")

print("-" * 80)
print("\nThese countries show CHRONIC underfunding patterns requiring sustained attention.")

## 6. Conclusions & Recommendations

### Key conclusions (Geo-Insight)

1. **Mismatch is systematic**: higher humanitarian need does not reliably translate into higher requested resources per person in need.
2. **Context matters**: under-allocation patterns vary by **region**, **crisis driver/type**, and our **crisis duration proxy** (see the driver analysis + feature importance section).
3. **Sector gaps concentrate inside underserved crises**: the sector coverage heatmap helps move from “which countries?” to “which clusters to prioritize?”

### Key conclusions (Challenge 1: Smart targeting validation)

1. **CPB is heavy-tailed by nature**, but **tiny beneficiary denominators** can create extreme CPB values that should be treated as a data-quality / reporting risk.
2. **Within-cluster benchmarking** produces a defensible outlier queue and a standardized cluster efficiency scorecard.
3. An interpretable regression shows which project attributes are most associated with higher CPB (after controlling for cluster and scale).

### Actionable recommendations for UN decision-makers

- **Quarterly monitoring (Geo-Insight)**: recompute the country ranking, highlight persistent top-N underserved crises, and use the feature analysis (driver/type/duration) for advocacy narratives.
- **Within-country prioritization**: use the sector coverage heatmap to focus on clusters with the lowest Targeted / In Need.
- **Audit queue (Challenge 1)**: triage CPB outliers with a two-step workflow:
  1) data-quality review for very small denominators / missing beneficiary breakdowns
  2) programmatic review for true high-cost-per-beneficiary cases

### Limitations (what could change conclusions)

- **HRP `revisedRequirements` ≠ disbursed funding**: we measure requested resources as a consistent proxy.
- **INFORM lag**: we proxy 2026 severity with the latest available year.
- **Cluster inference is keyword-based**: it is transparent and auditable, but not perfect—spot-check classifications.
- **Beneficiary fields can be missing/low**: ratios require thresholds and careful interpretation.

### Dashboard access

Explore the interactive dashboard at: **[datathon-2026.vercel.app](https://datathon-2026.vercel.app)**


## Part B — Smart Beneficiary Targeting Validation (Challenge 1)

This section answers: **Which projects look inefficient given their beneficiary counts, and what factors drive CPB?**

**Targets covered (Challenge 1):**
- CPB outlier detection with denominator guardrails
- Cluster efficiency benchmarking
- Explainable drivers of CPB (Ridge + XGBoost/Gradient Boosting)

### 7. Smart Beneficiary Targeting Validation (Challenge 1)

Goal: flag **statistical outliers in cost-per-beneficiary (CPB)** and build an **efficiency benchmarking framework** across humanitarian clusters.

### Feature engineering (project-level)
We engineer explainable features from the CBPF Projects dataset:
- **Budget**: `budget_usd` and `log10_budget`
- **Beneficiaries**: `beneficiaries_total = Men + Women + Boys + Girls` and `log10_beneficiaries`
- **Cost-per-beneficiary**: `cpb_usd_per_beneficiary = budget_usd / beneficiaries_total` (log-scaled for heavy tails)
- **Support cost share**: `TotalSupportCost / Budget` (when available)
- **Duration**: derived from `ProjectDuration` (months) and dates (when parsable)
- **Country exposure**: `beneficiary_share_of_country_pop`
- **Program metadata**: `OrganizationType`, `AllocationTypeCategory`
- **Cluster**: inferred via **transparent keyword rules** on project text fields (title/summary/activities/etc.)

### Guardrails (mandatory)
- CPB is extremely sensitive to tiny denominators: we **separate data-quality risks** (e.g., beneficiaries < 50) from true inefficiency signals.
- We run outlier detection **within-cluster** to avoid mixing incomparable program types.


In [ ]:
# Load Challenge 1 data (CBPF Projects) + engineer CPB features

import re
import zipfile
from pathlib import Path
from typing import Dict, List, Tuple

# Locate zip (supports running from repo root or /notebooks)
ZIP_CANDIDATES = [
    Path("data/project_targeting/project_targeting_data.zip"),
    Path("../data/project_targeting/project_targeting_data.zip"),
    Path("../../data/project_targeting/project_targeting_data.zip"),
]

DATA_ZIP_PATH = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if DATA_ZIP_PATH is None:
    raise FileNotFoundError("Could not find project_targeting_data.zip. Tried: " + ", ".join(map(str, ZIP_CANDIDATES)))

print("Using data zip:", DATA_ZIP_PATH.resolve())

with zipfile.ZipFile(DATA_ZIP_PATH) as z:
    projects_raw = pd.read_csv(
        z.open("project_targeting_data/Data_ Country Based Pooled Funds (CBPF) - Projects.csv"),
        encoding="utf-8-sig",
        low_memory=False,
    )
    contrib_raw = pd.read_csv(
        z.open("project_targeting_data/Data_ Country Based Pooled Funds (CBPF) - Contributions.csv"),
        encoding="utf-8-sig",
        low_memory=False,
    )
    pop0_raw = pd.read_csv(
        z.open("project_targeting_data/cod_population_admin0.csv"),
        encoding="utf-8-sig",
        low_memory=False,
    )

# Population table: keep latest total population per ISO3 (T_TL, all/all)
pop0 = pop0_raw.copy()
pop0 = pop0[~pop0["ISO3"].astype(str).str.startswith("#")].copy()

pop0["Population"] = pd.to_numeric(pop0["Population"], errors="coerce")
pop0["Reference_year"] = pd.to_numeric(pop0["Reference_year"], errors="coerce")

pop_country_total = pop0[
    (pop0["Population_group"] == "T_TL")
    & (pop0["Gender"] == "all")
    & (pop0["Age_range"] == "all")
].copy()

pop_country_total = (
    pop_country_total.sort_values(["ISO3", "Reference_year"])
    .groupby("ISO3", as_index=False)
    .tail(1)[["ISO3", "Reference_year", "Population", "Country"]]
    .rename(
        columns={
            "Reference_year": "population_year",
            "Population": "country_population",
            "Country": "country_name",
        }
    )
)

# Map pooled fund → ISO3 using contributions metadata
contrib = contrib_raw.copy()
fund_map = contrib[["PooledFundId", "PooledFundName", "PooledFundCodeAbbrv"]].drop_duplicates().copy()
fund_map["ISO3"] = fund_map["PooledFundCodeAbbrv"].astype(str).str[:3]
fund_map.loc[~fund_map["ISO3"].str.fullmatch(r"[A-Z]{3}"), "ISO3"] = np.nan

# Projects table
projects = projects_raw.copy()
projects["budget_usd"] = pd.to_numeric(projects["Budget"], errors="coerce")
projects["support_cost_usd"] = pd.to_numeric(projects.get("TotalSupportCost"), errors="coerce")
projects["direct_cost_usd"] = pd.to_numeric(projects.get("TotalDirectCost"), errors="coerce")

for col in ["Men", "Women", "Boys", "Girls"]:
    projects[col] = pd.to_numeric(projects[col], errors="coerce")

projects["beneficiaries_total"] = projects[["Men", "Women", "Boys", "Girls"]].fillna(0).sum(axis=1)

# Guardrails for ratios
projects.loc[projects["budget_usd"] <= 0, "budget_usd"] = np.nan
projects.loc[projects["beneficiaries_total"] <= 0, "beneficiaries_total"] = np.nan

projects["cpb_usd_per_beneficiary"] = projects["budget_usd"] / projects["beneficiaries_total"]
projects.loc[projects["cpb_usd_per_beneficiary"] <= 0, "cpb_usd_per_beneficiary"] = np.nan
projects["log10_cpb"] = np.log10(projects["cpb_usd_per_beneficiary"])

# Support cost share (when available)
projects["support_cost_share"] = projects["support_cost_usd"] / projects["budget_usd"]
projects.loc[~np.isfinite(projects["support_cost_share"]), "support_cost_share"] = np.nan

# Duration (months) from ProjectDuration string (e.g., "12 Months")
def parse_duration_months(x) -> float:
    m = re.search(r"(\d+)", str(x))
    return float(m.group(1)) if m else np.nan

projects["duration_months"] = projects["ProjectDuration"].apply(parse_duration_months)

# Enrich with ISO3 and country population
projects = projects.merge(fund_map[["PooledFundId", "ISO3"]], on="PooledFundId", how="left")
projects = projects.merge(pop_country_total, left_on="ISO3", right_on="ISO3", how="left")

projects["beneficiary_share_of_country_pop"] = projects["beneficiaries_total"] / projects["country_population"]
projects.loc[~np.isfinite(projects["beneficiary_share_of_country_pop"]), "beneficiary_share_of_country_pop"] = np.nan

# ----------------------------
# Cluster inference (explainable keyword mapping)
# ----------------------------
TEXT_COLS = [
    "ProjectTitle",
    "ProjectSummary",
    "Activities",
    "HumanitarianContext",
    "DescriptionOfBeneficiaries",
    "NeedsAssessment",
]

CLUSTER_KEYWORDS: Dict[str, List[str]] = {
    "Health": ["health", "medical", "clinic", "hospital", "trauma", "referral", "reproductive", "maternal", "immuniz", "vaccine", "cholera", "measles", "malaria"],
    "WASH": ["wash", "water", "sanitation", "hygiene", "latrine", "borehole", "well", "chlorin", "soap", "handwashing"],
    "Education": ["education", "school", "teacher", "learning", "classroom", "student", "literacy"],
    "Nutrition": ["nutrition", "malnutrition", "sam", "mam", "iycf", "cmam", "therapeutic", "feeding", "micronutrient"],
    "Food Security": ["food security", "livelihood", "agric", "seed", "fertilizer", "livestock", "voucher", "cash for work"],
    "Shelter/NFI": ["shelter", "nfi", "non-food", "tent", "tarpaulin", "winterization", "housing", "repair"],
    "Protection": ["protection", "gbv", "gender-based violence", "child protection", "psychosocial", "legal", "safe space", "case management"],
    "Mine Action": ["mine", "demining", "uxo", "explosive", "ordnance"],
}


def infer_cluster(text: str) -> Tuple[str, str, str]:
    """Return (primary_cluster, all_clusters, evidence_string)."""
    t = (text or "").lower()
    hits: Dict[str, List[str]] = {}
    for cluster, kws in CLUSTER_KEYWORDS.items():
        matched = [kw for kw in kws if kw in t]
        if matched:
            hits[cluster] = matched
    if not hits:
        return "Uncategorized", "", ""

    best_cluster = sorted(hits.items(), key=lambda kv: len(kv[1]), reverse=True)[0][0]
    all_clusters = "; ".join(sorted(hits.keys()))

    ev = f"{best_cluster} (keywords: {', '.join(hits[best_cluster][:5])})"
    if len(hits) > 1:
        others = [c for c in sorted(hits.keys()) if c != best_cluster]
        ev += f"; also matched: {', '.join(others)}"

    return best_cluster, all_clusters, ev


projects["project_text"] = projects[TEXT_COLS].fillna("").astype(str).agg(" ".join, axis=1)
cluster_cols = projects["project_text"].apply(lambda t: pd.Series(infer_cluster(t)))
cluster_cols.columns = ["cluster_primary", "cluster_all", "cluster_evidence"]
projects = pd.concat([projects, cluster_cols], axis=1)

print("Rows:", len(projects))
print("CPB available:", f"{projects['cpb_usd_per_beneficiary'].notna().mean():.1%}")
print("Population merged:", f"{projects['country_population'].notna().mean():.1%}")
projects["cluster_primary"].value_counts().head(12)


In [ ]:
# Challenge 1 EDA: denominator risk + CPB distributions

import matplotlib.ticker as mtick

MIN_BEN_REVIEW = 50

# Missingness snapshot (high-signal fields)
key_cols = [
    "budget_usd",
    "beneficiaries_total",
    "cpb_usd_per_beneficiary",
    "log10_cpb",
    "support_cost_share",
    "duration_months",
    "ISO3",
    "cluster_primary",
]
key_cols = [c for c in key_cols if c in projects.columns]
missing_rate = projects[key_cols].isna().mean().sort_values(ascending=False)
print("Missingness (share):")
print(missing_rate.to_string())

# Denominator risk
b = projects["beneficiaries_total"].dropna()
print("\nBeneficiary denominator risk:")
print(f"% beneficiaries < {MIN_BEN_REVIEW}: {(b < MIN_BEN_REVIEW).mean():.1%}")
print(f"% beneficiaries < 100: {(b < 100).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Beneficiaries distribution (log10)
ax = axes[0]
log_b = np.log10(b)
if sns is not None:
    sns.histplot(log_b, bins=45, kde=True, ax=ax, color="#3b82f6", alpha=0.6)
else:
    ax.hist(log_b, bins=45, color="#3b82f6", alpha=0.6)

ax.axvline(np.log10(MIN_BEN_REVIEW), color="#ef4444", linestyle="--", linewidth=2, label=str(MIN_BEN_REVIEW))
ax.axvline(np.log10(100), color="#f59e0b", linestyle="--", linewidth=2, label="100")
ax.set_title("Planned beneficiaries (log scale)\nSmall denominators inflate CPB")
ax.set_xlabel("log10(beneficiaries)")
ax.set_ylabel("Projects")
ax.legend(title="Threshold")

# CPB distribution (log10)
ax = axes[1]
viz_all = projects.dropna(subset=["log10_cpb"]).copy()
if sns is not None:
    sns.histplot(viz_all["log10_cpb"], bins=55, kde=True, ax=ax, color="#10b981", alpha=0.6)
else:
    ax.hist(viz_all["log10_cpb"], bins=55, color="#10b981", alpha=0.6)

med = viz_all["log10_cpb"].median()
p90 = viz_all["log10_cpb"].quantile(0.90)
ax.axvline(med, color="#111827", linewidth=2, label=f"Median ${10**med:,.0f}")
ax.axvline(p90, color="#ef4444", linestyle="--", linewidth=2, label=f"90th pct ${10**p90:,.0f}")
ax.set_title("Cost-per-beneficiary distribution (log scale)\nHeavy tails are expected")
ax.set_xlabel("log10(USD per beneficiary)")
ax.set_ylabel("Projects")
ax.legend()

plt.tight_layout()
plt.show()

# Cluster comparisons (exclude Uncategorized; require enough rows)
viz_df = projects.dropna(subset=["log10_cpb", "cluster_primary"]).copy()
viz_df = viz_df[viz_df["cluster_primary"] != "Uncategorized"].copy()

cluster_n = viz_df["cluster_primary"].value_counts()
clusters_keep = cluster_n[cluster_n >= 30].index
viz_df = viz_df[viz_df["cluster_primary"].isin(clusters_keep)].copy()

order = viz_df.groupby("cluster_primary")["log10_cpb"].median().sort_values().index

fig, ax = plt.subplots(figsize=(12, 6))
if sns is not None:
    sns.boxplot(data=viz_df, x="log10_cpb", y="cluster_primary", order=order, ax=ax, color="#93c5fd")
else:
    # fallback: show median bars
    med = viz_df.groupby("cluster_primary")["log10_cpb"].median().loc[order]
    ax.barh(med.index, med.values, color="#93c5fd", edgecolor="black")

ax.set_title("Cluster CPB comparison (log scale)\nWithin-cluster benchmarking avoids apples-to-oranges")
ax.set_xlabel("log10(USD per beneficiary)")
ax.set_ylabel("Cluster (inferred)")
plt.tight_layout()
plt.show()


In [ ]:
# Outlier detection (explainable, within-cluster) + standardized efficiency framework

MIN_CLUSTER_N = 30
Z_THRESH = 3.0
IQR_K = 1.5
PCT_THRESH = 0.99
MIN_BEN_REVIEW = 50

from pathlib import Path


def show(df, n=15):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


model_df = projects.dropna(subset=["log10_cpb", "cluster_primary"]).copy()
model_df = model_df[model_df["cluster_primary"] != "Uncategorized"].copy()

# Keep only clusters with enough samples to estimate a distribution
cluster_n = model_df["cluster_primary"].value_counts()
eligible_clusters = cluster_n[cluster_n >= MIN_CLUSTER_N].index
model_df = model_df[model_df["cluster_primary"].isin(eligible_clusters)].copy()

cluster_stats = (
    model_df.groupby("cluster_primary")["log10_cpb"]
    .agg(
        n="count",
        mean_log10="mean",
        std_log10="std",
        median_log10="median",
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75),
    )
    .reset_index()
)
cluster_stats["iqr"] = cluster_stats["q3"] - cluster_stats["q1"]
cluster_stats["upper_fence"] = cluster_stats["q3"] + IQR_K * cluster_stats["iqr"]

model_df = model_df.merge(cluster_stats, on="cluster_primary", how="left")

# Z-score in log space
model_df["z_log10_cpb"] = (model_df["log10_cpb"] - model_df["mean_log10"]) / model_df["std_log10"]
model_df.loc[~np.isfinite(model_df["z_log10_cpb"]), "z_log10_cpb"] = 0.0

# Percentile rank (within cluster)
model_df["cpb_percentile_in_cluster"] = model_df.groupby("cluster_primary")["log10_cpb"].rank(pct=True)

# Flags (high CPB outliers)
model_df["flag_iqr_high"] = model_df["log10_cpb"] > model_df["upper_fence"]
model_df["flag_z_high"] = model_df["z_log10_cpb"] >= Z_THRESH
model_df["flag_pct_high"] = model_df["cpb_percentile_in_cluster"] >= PCT_THRESH

# Data-quality flags
model_df["flag_small_denominator"] = model_df["beneficiaries_total"] < MIN_BEN_REVIEW
model_df["flag_beneficiaries_gt_country_pop"] = (
    (model_df["country_population"].notna())
    & (model_df["beneficiaries_total"].notna())
    & (model_df["beneficiaries_total"] > model_df["country_population"])
)

model_df["flag_any"] = model_df[["flag_iqr_high", "flag_z_high", "flag_pct_high"]].any(axis=1)

# Explainable reason string

def build_reason(r) -> str:
    reasons = []
    if r.get("flag_small_denominator"):
        reasons.append(f"Very small planned beneficiaries (<{MIN_BEN_REVIEW})")
    if r.get("flag_iqr_high"):
        reasons.append("Above IQR upper fence (within cluster)")
    if r.get("flag_z_high"):
        reasons.append(f"High Z-score (>= {Z_THRESH:.1f}) in log10 CPB")
    if r.get("flag_pct_high"):
        reasons.append(f">= {PCT_THRESH:.0%} percentile CPB in cluster")
    if r.get("flag_beneficiaries_gt_country_pop"):
        reasons.append("Beneficiaries exceed country population (data issue)")
    return "; ".join(reasons)

model_df["outlier_reason"] = model_df.apply(build_reason, axis=1)

outliers = (
    model_df[model_df["flag_any"]]
    .sort_values(["cluster_primary", "log10_cpb"], ascending=[True, False])
    .copy()
)

print("Eligible clusters:", len(eligible_clusters))
print("Projects evaluated:", len(model_df))
print("Flagged outliers:", len(outliers), f"({len(outliers)/len(model_df):.1%} of evaluated)")

# Standardized cluster efficiency framework
cluster_summary = (
    model_df.groupby("cluster_primary")
    .agg(
        n_projects=("cpb_usd_per_beneficiary", "count"),
        median_cpb_usd=("cpb_usd_per_beneficiary", "median"),
        p10_cpb_usd=("cpb_usd_per_beneficiary", lambda s: s.quantile(0.10)),
        p90_cpb_usd=("cpb_usd_per_beneficiary", lambda s: s.quantile(0.90)),
        outlier_rate=("flag_any", "mean"),
    )
    .reset_index()
)

# Score clusters: lower median CPB is better, penalize high outlier rate
cluster_summary["median_log10_cpb"] = np.log10(cluster_summary["median_cpb_usd"])
min_log = cluster_summary["median_log10_cpb"].min()
max_log = cluster_summary["median_log10_cpb"].max()

cluster_summary["efficiency_score_raw"] = (
    100.0
    if max_log == min_log
    else 100 * (1 - (cluster_summary["median_log10_cpb"] - min_log) / (max_log - min_log))
)

cluster_summary["efficiency_score"] = (
    cluster_summary["efficiency_score_raw"] - 25 * cluster_summary["outlier_rate"]
).clip(0, 100)

cluster_summary = cluster_summary.sort_values(
    ["efficiency_score", "n_projects"], ascending=[False, False]
).reset_index(drop=True)
cluster_summary["rank"] = np.arange(1, len(cluster_summary) + 1)

print("\nTop flagged outliers (showing 10):")
show(
    outliers[[
        "cluster_primary",
        "PooledFundName",
        "AllocationYear",
        "OrganizationName",
        "ProjectTitle",
        "budget_usd",
        "beneficiaries_total",
        "cpb_usd_per_beneficiary",
        "z_log10_cpb",
        "cpb_percentile_in_cluster",
        "outlier_reason",
        "cluster_evidence",
    ]].head(10),
    n=10,
)

print("\nCluster efficiency framework:")
show(cluster_summary[["rank", "cluster_primary", "n_projects", "median_cpb_usd", "p10_cpb_usd", "p90_cpb_usd", "outlier_rate", "efficiency_score"]], n=20)

# Optional: save outputs for officers to review
REPO_ROOT = DATA_ZIP_PATH.resolve().parents[2]
OUT_DIR = REPO_ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

outliers_export = outliers[[
    "cluster_primary",
    "PooledFundName",
    "AllocationYear",
    "OrganizationName",
    "ProjectTitle",
    "budget_usd",
    "beneficiaries_total",
    "cpb_usd_per_beneficiary",
    "cpb_percentile_in_cluster",
    "z_log10_cpb",
    "flag_iqr_high",
    "flag_z_high",
    "flag_pct_high",
    "flag_small_denominator",
    "flag_beneficiaries_gt_country_pop",
    "outlier_reason",
    "cluster_evidence",
]]

outliers_export.to_csv(OUT_DIR / "challenge1_outlier_projects.csv", index=False)
cluster_summary.to_csv(OUT_DIR / "challenge1_cluster_efficiency_framework.csv", index=False)

print("\nWrote:", OUT_DIR / "challenge1_outlier_projects.csv")
print("Wrote:", OUT_DIR / "challenge1_cluster_efficiency_framework.csv")


In [ ]:
# Feature importance (Challenge 1): what drives cost-per-beneficiary?

try:
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import Ridge
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import r2_score, mean_absolute_error
except Exception as e:
    raise ImportError(
        "scikit-learn is required for this section. In Databricks: `%pip install -q scikit-learn`"
    ) from e

feat_df = model_df.dropna(subset=["budget_usd", "beneficiaries_total", "log10_cpb"]).copy()

# Optional: exclude tiny denominators for the 'efficiency' model (separate from data-quality review list)
feat_df_eff = feat_df[feat_df["beneficiaries_total"] >= MIN_BEN_REVIEW].copy()

for df_name, df in [("all_evaluable", feat_df), (f"beneficiaries>={MIN_BEN_REVIEW}", feat_df_eff)]:
    df = df.copy()
    df["log10_budget"] = np.log10(df["budget_usd"].where(df["budget_usd"] > 0))
    df["log10_beneficiaries"] = np.log10(df["beneficiaries_total"].where(df["beneficiaries_total"] > 0))

    target = df["log10_cpb"]

    num_features = [
        "log10_budget",
        "log10_beneficiaries",
        "support_cost_share",
        "duration_months",
        "beneficiary_share_of_country_pop",
    ]
    cat_features = [
        "cluster_primary",
        "OrganizationType",
        "AllocationTypeCategory",
    ]

    num_features = [c for c in num_features if c in df.columns]
    cat_features = [c for c in cat_features if c in df.columns]

    X = df[num_features + cat_features]

    X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=0.25, random_state=42)

    pre = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
                num_features,
            ),
            (
                "cat",
                Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]),
                cat_features,
            ),
        ]
    )

    pipe = Pipeline(steps=[("pre", pre), ("model", Ridge(alpha=1.0))])
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)
    print("\n=== Challenge 1 model:", df_name, "===")
    print("Rows:", len(df), "| Features:", len(num_features) + len(cat_features))
    print(f"Test R^2: {r2_score(y_test, pred):.3f} | Test MAE: {mean_absolute_error(y_test, pred):.3f}")

    # Coefficients
    feature_names = []
    feature_names.extend(num_features)
    if cat_features:
        ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
        feature_names.extend(ohe.get_feature_names_out(cat_features).tolist())

    coefs = pipe.named_steps["model"].coef_
    coef_df = (
        pd.DataFrame({"feature": feature_names, "coef": coefs})
        .assign(abs_coef=lambda d: d["coef"].abs())
        .sort_values("abs_coef", ascending=False)
    )

    top_k = 18
    plot_coef = coef_df.head(top_k).sort_values("coef")

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = ["#dc2626" if v < 0 else "#16a34a" for v in plot_coef["coef"]]
    ax.barh(plot_coef["feature"], plot_coef["coef"], color=colors, edgecolor="black", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Top factors associated with log10(CPB) — {df_name}\n(positive = higher CPB; negative = lower CPB)")
    ax.set_xlabel("Ridge coefficient (standardized numeric features)")
    plt.tight_layout()
    plt.show()

    # Nonlinear model (XGBoost if available, otherwise GradientBoosting)
    try:
        from xgboost import XGBRegressor

        tree_model = XGBRegressor(
            n_estimators=350,
            max_depth=4,
            learning_rate=0.06,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
        )
        tree_model_name = "XGBoost"
        tree_ohe_sparse = True
    except Exception:
        from sklearn.ensemble import GradientBoostingRegressor

        tree_model = GradientBoostingRegressor(random_state=42)
        tree_model_name = "GradientBoosting"
        tree_ohe_sparse = False

    try:
        tree_ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=tree_ohe_sparse)
    except TypeError:
        tree_ohe = OneHotEncoder(handle_unknown="ignore", sparse=tree_ohe_sparse)

    pre_tree = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_features),
            (
                "cat",
                Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", tree_ohe)]),
                cat_features,
            ),
        ]
    )

    tree_pipe = Pipeline(steps=[("pre", pre_tree), ("model", tree_model)])
    tree_pipe.fit(X_train, y_train)
    tree_pred = tree_pipe.predict(X_test)

    print(f"Model: {tree_model_name} (nonlinear) — {df_name}")
    print(f"Test R^2: {r2_score(y_test, tree_pred):.3f} | Test MAE: {mean_absolute_error(y_test, tree_pred):.3f}")

    if hasattr(tree_pipe.named_steps["model"], "feature_importances_"):
        feature_names_tree = []
        feature_names_tree.extend(num_features)

        if cat_features:
            ohe_tree = tree_pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
            feature_names_tree.extend(ohe_tree.get_feature_names_out(cat_features).tolist())

        importances = tree_pipe.named_steps["model"].feature_importances_
        imp_df = (
            pd.DataFrame({"feature": feature_names_tree, "importance": importances})
            .sort_values("importance", ascending=False)
            .head(12)
            .sort_values("importance")
        )

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(imp_df["feature"], imp_df["importance"], color="#2563eb", edgecolor="black", linewidth=0.5)
        ax.set_title(f"Top nonlinear drivers of log10(CPB) — {df_name}")
        ax.set_xlabel("Feature importance")
        plt.tight_layout()
        plt.show()

print("\nInterpretation notes:")
print("- CPB is mechanically driven by budget and beneficiary counts; log transforms make effects readable.")
print("- Cluster coefficients show systematic differences across program types (after controlling for size/cost structure).")
print("- Nonlinear model results are predictive checks (not causal).")
